In [1]:
import requests
import pandas as pd
import math
import folium

from getpass import getpass

In [2]:
# Configurações da API
BASE_URL = "https://api.olhovivo.sptrans.com.br/v2.1"

# Distâncias em metros
RAIO_PARADAS = 500
RAIO_VEICULOS = 3000

# Limite de linhas analisadas
LIMITE_CANDIDATAS = 100

# Sessão de comunicação com a API
sessao_api = requests.Session()

print("Configurações carregadas com sucesso!")

Configurações carregadas com sucesso!


In [3]:
def autenticar():

    token = getpass("Digite seu token SPTrans: ")

    resposta = sessao_api.post(
        f"{BASE_URL}/Login/Autenticar",
        params={"token": token},
        timeout=20
    )

    resposta.raise_for_status()

    if resposta.json() is True:
        print("Autenticação realizada com sucesso!")
        return True

    raise Exception("Não foi possível autenticar na SPTrans.")

In [4]:
autenticar()

Autenticação realizada com sucesso!


True

In [5]:
def calcular_distancia(lat1, lon1, lat2, lon2):

    raio_terra = 6371000

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)

    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    diferenca_lat = lat2 - lat1
    diferenca_lon = lon2 - lon1

    a = (
        math.sin(diferenca_lat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(diferenca_lon / 2) ** 2
    )

    distancia = 2 * raio_terra * math.asin(
        math.sqrt(a)
    )

    return distancia

In [6]:
def consultar_veiculos():

    resposta = sessao_api.get(
        f"{BASE_URL}/Posicao",
        timeout=60
    )

    resposta.raise_for_status()

    dados = resposta.json()

    if not isinstance(dados, dict):
        raise ValueError("Resposta inválida da SPTrans.")

    registros = []

    for linha in dados.get("l") or []:

        for veiculo in linha.get("vs") or []:

            if veiculo.get("py") is None or veiculo.get("px") is None:
                continue

            registros.append({
                "Codigo_linha": linha["cl"],
                "Letreiro": linha["c"],
                "Sentido": linha["sl"],
                "Prefixo": veiculo.get("p"),
                "Latitude": veiculo["py"],
                "Longitude": veiculo["px"],
                "Destino": linha.get("lt0"),
                "Origem": linha.get("lt1")
            })

    return pd.DataFrame(registros)


# Consultar veículos
df_veiculos = consultar_veiculos()

print("Veículos encontrados:", len(df_veiculos))

display(df_veiculos.head())

Veículos encontrados: 6670


,Codigo_linha,Letreiro,Sentido,Prefixo,Latitude,Longitude,Destino,Origem
0,867,177Y-10,1,21238,-23.535530,-46.674306,PINHEIROS,METRÔ BARRA FUNDA
1,867,177Y-10,1,21246,-23.527352,-46.666758,PINHEIROS,METRÔ BARRA FUNDA
2,1629,574W-10,1,55558,-23.612674,-46.494038,METRÔ BELÉM,JD. WALKIRIA
3,1629,574W-10,1,55412,-23.619152,-46.486465,METRÔ BELÉM,JD. WALKIRIA
4,1629,574W-10,1,55472,-23.594074,-46.519174,METRÔ BELÉM,JD. WALKIRIA


In [7]:
# Entrada das coordenadas de origem e destino

#-23.534564
#-46.654302
#-23.525799
#-46.679251

#-23.527606
#-46.671514
#-23.522339
#-46.696246

print("SISTEMA DE RECOMENDAÇÃO DE TRANSPORTE")
print("\nInforme as coordenadas geográficas.")

try:
    print("\n--- ORIGEM ---")

    latitude_origem = float(
        input("Latitude da origem: ").strip().replace(",", ".")
    )

    longitude_origem = float(
        input("Longitude da origem: ").strip().replace(",", ".")
    )

    print("\n--- DESTINO ---")

    latitude_destino = float(
        input("Latitude do destino: ").strip().replace(",", ".")
    )

    longitude_destino = float(
        input("Longitude do destino: ").strip().replace(",", ".")
    )

except ValueError:
    raise ValueError(
        "Coordenadas inválidas. Informe valores numéricos."
    )

# Validar os limites geográficos
if not (-90 <= latitude_origem <= 90):
    raise ValueError("Latitude da origem inválida.")

if not (-180 <= longitude_origem <= 180):
    raise ValueError("Longitude da origem inválida.")

if not (-90 <= latitude_destino <= 90):
    raise ValueError("Latitude do destino inválida.")

if not (-180 <= longitude_destino <= 180):
    raise ValueError("Longitude do destino inválida.")

# Organizar as coordenadas
origem = (latitude_origem, longitude_origem)
destino = (latitude_destino, longitude_destino)

if origem == destino:
    raise ValueError("Origem e destino não podem ser iguais.")

print("\nCoordenadas recebidas e validadas!")
print("Origem:", origem)
print("Destino:", destino)

# Calcular distância de cada veículo até a origem
df_veiculos["Distancia_origem"] = df_veiculos.apply(
    lambda v: calcular_distancia(
        v["Latitude"], v["Longitude"],
        origem[0], origem[1]
    ),
    axis=1
)

# Calcular distância de cada veículo até o destino
df_veiculos["Distancia_destino"] = df_veiculos.apply(
    lambda v: calcular_distancia(
        v["Latitude"], v["Longitude"],
        destino[0], destino[1]
    ),
    axis=1
)

# Resumir as informações por linha
df_resumo = df_veiculos.groupby("Codigo_linha").agg(
    Letreiro=("Letreiro", "first"),
    Sentido=("Sentido", "first"),
    Origem=("Origem", "first"),
    Destino=("Destino", "first"),
    Quantidade_veiculos=("Prefixo", "count"),
    Menor_distancia_origem=("Distancia_origem", "min"),
    Menor_distancia_destino=("Distancia_destino", "min")
).reset_index()

# Identificar linhas próximas de pelo menos um dos pontos
df_candidatas = df_resumo[
    (df_resumo["Menor_distancia_origem"] <= RAIO_VEICULOS) |
    (df_resumo["Menor_distancia_destino"] <= RAIO_VEICULOS)
].copy()

# Priorizar linhas com veículos próximos dos dois pontos
df_candidatas["Pontuacao"] = (
    df_candidatas["Menor_distancia_origem"] +
    df_candidatas["Menor_distancia_destino"]
)

df_candidatas = df_candidatas.sort_values("Pontuacao")

# Limitar a quantidade de linhas analisadas
total_candidatas = len(df_candidatas)

df_candidatas = df_candidatas.head(
    LIMITE_CANDIDATAS
).copy()

print("Total de linhas candidatas:", total_candidatas)
print("Linhas selecionadas para análise:", len(df_candidatas))

display(df_candidatas.head(15))

SISTEMA DE RECOMENDAÇÃO DE TRANSPORTE

Informe as coordenadas geográficas.

--- ORIGEM ---

--- DESTINO ---

Coordenadas recebidas e validadas!
Origem: (-23.534564, -46.654302)
Destino: (-23.525799, -46.679251)
Total de linhas candidatas: 275
Linhas selecionadas para análise: 100


,Codigo_linha,Letreiro,Sentido,Origem,Destino,Quantidade_veiculos,Menor_distancia_origem,Menor_distancia_destino,Pontuacao
573,1265,8400-10,1,TERM. PIRITUBA,PÇA. RAMOS DE AZEVEDO,7,331.064815,440.442647,771.507462
173,438,8622-10,1,MORRO DOCE,PÇA. RAMOS DE AZEVEDO,5,1230.616570,568.978074,1799.594643
1122,33230,9501-10,2,TERM. CACHOEIRINHA,LGO. DO PAISSANDÚ,3,965.396246,899.600918,1864.997163
174,443,8677-10,1,JD. LÍBANO,LGO. DO PAISSANDU,3,1871.878131,25.943089,1897.821220
237,609,875A-10,1,AEROPORTO,PERDIZES,4,750.648844,1185.160467,1935.809311
1546,34109,875P-10,2,METRÔ BARRA FUNDA,METRÔ ANA ROSA,2,672.911640,1370.021156,2042.932797
1263,33596,9785-10,2,VL. TEREZINHA,METRÔ BARRA FUNDA,9,1486.513500,638.629394,2125.142894
1851,35274,8000-1,2,TERM. LAPA,PÇA. RAMOS DE AZEVEDO,3,714.075677,1762.673128,2476.748805
1126,33237,719R-10,2,RIO PEQUENO,METRÔ BARRA FUNDA,3,1298.160764,1327.700368,2625.861132
344,867,177Y-10,1,METRÔ BARRA FUNDA,PINHEIROS,2,1501.817346,1193.666160,2695.483505


In [8]:
# Armazenar as paradas consultadas
cache_paradas = {}

In [9]:
resultados_paradas = []
erros_consulta = []

for _, linha in df_candidatas.iterrows():

    codigo = int(linha["Codigo_linha"])

    try:
        # Consultar paradas da linha
        if codigo not in cache_paradas:

            resposta = sessao_api.get(
                f"{BASE_URL}/Parada/BuscarParadasPorLinha",
                params={"codigoLinha": codigo},
                timeout=30
            )

            # Interromper se a autenticação expirar
            if resposta.status_code == 401:
                print("Autenticação expirada. Renove a sessão.")
                break

            resposta.raise_for_status()

            cache_paradas[codigo] = resposta.json()

        paradas = cache_paradas[codigo]

        if not isinstance(paradas, list) or not paradas:
            continue

        df = pd.DataFrame(paradas)

        df["py"] = pd.to_numeric(df["py"], errors="coerce")
        df["px"] = pd.to_numeric(df["px"], errors="coerce")

        df = df.dropna(subset=["py", "px"])

        if df.empty:
            continue

        # Calcular distâncias até a origem
        df["Distancia_origem"] = df.apply(
            lambda p: calcular_distancia(
                *origem, p["py"], p["px"]
            ),
            axis=1
        )

        # Calcular distâncias até o destino
        df["Distancia_destino"] = df.apply(
            lambda p: calcular_distancia(
                *destino, p["py"], p["px"]
            ),
            axis=1
        )

        # Encontrar as paradas mais próximas
        embarque = df.loc[df["Distancia_origem"].idxmin()]
        desembarque = df.loc[df["Distancia_destino"].idxmin()]

        resultados_paradas.append({
            "Codigo_linha": codigo,
            "Letreiro": linha["Letreiro"],
            "Sentido": linha["Sentido"],
            "Total_paradas": len(df),

            "Embarque": embarque["np"],
            "Codigo_embarque": embarque["cp"],
            "Distancia_embarque": round(
                embarque["Distancia_origem"], 2
            ),

            "Desembarque": desembarque["np"],
            "Codigo_desembarque": desembarque["cp"],
            "Distancia_desembarque": round(
                desembarque["Distancia_destino"], 2
            )
        })

    except requests.exceptions.RequestException as erro:
        erros_consulta.append({
            "Codigo_linha": codigo,
            "Erro": str(erro)
        })


# Organizar os resultados
df_analise_paradas = pd.DataFrame(resultados_paradas)

print("Linhas consultadas:", len(cache_paradas))
print("Linhas com dados válidos:", len(df_analise_paradas))
print("Erros encontrados:", len(erros_consulta))

display(df_analise_paradas)

Linhas consultadas: 100
Linhas com dados válidos: 69
Erros encontrados: 0


,Codigo_linha,Letreiro,Sentido,Total_paradas,Embarque,Codigo_embarque,Distancia_embarque,Desembarque,Codigo_desembarque,Distancia_desembarque
0,1265,8400-10,1,26,ANGELICA B/C,7014417,0.00,PARADA PALMEIRAS B/C,60016784,0.00
1,438,8622-10,1,13,ANGELICA B/C,7014417,0.00,PARADA PALMEIRAS B/C,60016784,0.00
2,33230,9501-10,2,11,EDUARDO PRADO C/B,700016619,890.40,INDUSTRIAL - C/B,60016885,833.02
3,443,8677-10,1,13,ANGELICA B/C,7014417,0.00,PARADA PALMEIRAS B/C,60016784,0.00
4,609,875A-10,1,6,ANGELICA C/B,7014416,20.89,ROSA E SILVA C/B,700006085,2347.33
...,...,...,...,...,...,...,...,...,...,...
64,1432,8600-10,1,22,EDUARDO PRADO - B/C,700016620,886.49,INDUSTRIAL - B/C,60016884,826.56
65,33075,874T-10,2,6,ROSA E SILVA C/B,700006086,369.40,SESC POMPEIA B/C,610014603,429.55
66,309,311C-10,1,6,INHOMERIM B/C,540014153,6448.29,INHOMERIM B/C,540014153,9157.68
67,32772,669A-10,2,13,NOTHMANN C/B,700016866,371.45,NOTHMANN C/B,700016866,3089.61


In [10]:
# Verificar quais linhas atendem aos dois pontos
df_linhas_compativeis = df_analise_paradas[
    (df_analise_paradas["Distancia_embarque"] <= RAIO_PARADAS) &
    (df_analise_paradas["Distancia_desembarque"] <= RAIO_PARADAS)
].copy()

# Calcular distância total de caminhada
df_linhas_compativeis["Distancia_caminhada"] = (
    df_linhas_compativeis["Distancia_embarque"] +
    df_linhas_compativeis["Distancia_desembarque"]
)

# Priorizar menor distância total
df_linhas_compativeis = df_linhas_compativeis.sort_values(
    "Distancia_caminhada"
)

print("Linhas que atendem à origem e ao destino:",
      len(df_linhas_compativeis))

display(df_linhas_compativeis)

Linhas que atendem à origem e ao destino: 18


,Codigo_linha,Letreiro,Sentido,Total_paradas,Embarque,Codigo_embarque,Distancia_embarque,Desembarque,Codigo_desembarque,Distancia_desembarque,Distancia_caminhada
0,1265,8400-10,1,26,ANGELICA B/C,7014417,0.00,PARADA PALMEIRAS B/C,60016784,0.00,0.00
1,438,8622-10,1,13,ANGELICA B/C,7014417,0.00,PARADA PALMEIRAS B/C,60016784,0.00,0.00
3,443,8677-10,1,13,ANGELICA B/C,7014417,0.00,PARADA PALMEIRAS B/C,60016784,0.00,0.00
10,1339,8615-10,1,13,ANGELICA B/C,7014417,0.00,PARADA PALMEIRAS B/C,60016784,0.00,0.00
59,591,978J-10,1,18,ANGELICA B/C,7014417,0.00,PARADA PALMEIRAS B/C,60016784,0.00,0.00
52,2520,1896-10,1,12,ANGELICA B/C,7014417,0.00,PARADA PALMEIRAS B/C,60016784,0.00,0.00
29,1273,8000-10,1,14,ANGELICA B/C,7014417,0.00,PARADA PALMEIRAS B/C,60016784,0.00,0.00
13,432,8594-10,1,23,ANGELICA B/C,7014417,0.00,PARADA PALMEIRAS B/C,60016784,0.00,0.00
11,33206,8622-10,2,13,ANGELICA C/B,7014416,20.89,PARADA PALMEIRAS C/B,60016783,29.53,50.42
9,34107,8615-10,2,14,ANGELICA C/B,7014416,20.89,PARADA PALMEIRAS C/B,60016783,29.53,50.42


In [11]:
import zipfile
import io
from datetime import date

# Endereço do GTFS da SPTrans
URL_GTFS = (
    "https://www.sptrans.com.br/"
    "umbraco/Surface/PerfilDesenvolvedor/BaixarGTFS"
)


def carregar_gtfs():

    resposta = requests.get(
        URL_GTFS,
        timeout=180
    )

    resposta.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resposta.content)) as arquivo:

        arquivos = {
            nome.split("/")[-1].lower(): nome
            for nome in arquivo.namelist()
        }

        necessarios = [
            "routes.txt",
            "trips.txt",
            "stops.txt",
            "stop_times.txt",
            "calendar.txt"
        ]

        for nome in necessarios:
            if nome not in arquivos:
                raise ValueError(
                    f"Arquivo GTFS ausente: {nome}"
                )

        rotas = pd.read_csv(
            arquivo.open(arquivos["routes.txt"]),
            dtype=str
        )

        viagens = pd.read_csv(
            arquivo.open(arquivos["trips.txt"]),
            dtype=str
        )

        paradas = pd.read_csv(
            arquivo.open(arquivos["stops.txt"]),
            dtype=str
        )

        horarios = pd.read_csv(
            arquivo.open(arquivos["stop_times.txt"]),
            dtype=str
        )

        calendario = pd.read_csv(
            arquivo.open(arquivos["calendar.txt"]),
            dtype=str
        )

    return rotas, viagens, paradas, horarios, calendario


# Carregar os dados
(
    df_gtfs_rotas,
    df_gtfs_viagens,
    df_gtfs_paradas,
    df_gtfs_horarios,
    df_gtfs_calendario
) = carregar_gtfs()

print("Dados GTFS carregados com sucesso!")

Dados GTFS carregados com sucesso!


In [12]:
from datetime import date

def obter_servicos_ativos(data_consulta=None):

    if data_consulta is None:
        data_consulta = date.today()

    dia_semana = [
        "monday", "tuesday", "wednesday",
        "thursday", "friday", "saturday", "sunday"
    ][data_consulta.weekday()]

    calendario = df_gtfs_calendario.copy()

    inicio = pd.to_datetime(
        calendario["start_date"],
        format="%Y%m%d"
    )

    fim = pd.to_datetime(
        calendario["end_date"],
        format="%Y%m%d"
    )

    data = pd.Timestamp(data_consulta)

    servicos = calendario[
        (inicio <= data) &
        (fim >= data) &
        (calendario[dia_semana] == "1")
    ]

    return set(servicos["service_id"])

In [13]:
def validar_trajeto_gtfs(linha, servicos_ativos):

    letreiro = str(linha["Letreiro"]).strip()

    codigo_embarque = str(int(linha["Codigo_embarque"]))
    codigo_desembarque = str(int(linha["Codigo_desembarque"]))

    # Localizar a rota correspondente
    rotas = df_gtfs_rotas[
        df_gtfs_rotas["route_short_name"].astype(str).str.strip()
        == letreiro
    ]

    if rotas.empty:
        return []

    codigos_rotas = rotas["route_id"].tolist()

    # Localizar viagens com serviços ativos
    viagens = df_gtfs_viagens[
        (df_gtfs_viagens["route_id"].isin(codigos_rotas)) &
        (df_gtfs_viagens["service_id"].isin(servicos_ativos))
    ]

    resultados = []

    for _, viagem in viagens.iterrows():

        horarios = df_gtfs_horarios[
            df_gtfs_horarios["trip_id"] == viagem["trip_id"]
        ].copy()

        horarios["stop_sequence"] = pd.to_numeric(
            horarios["stop_sequence"],
            errors="coerce"
        )

        # Encontrar as posições das duas paradas
        sequencias_embarque = horarios[
            horarios["stop_id"] == codigo_embarque
        ]["stop_sequence"].dropna().tolist()

        sequencias_desembarque = horarios[
            horarios["stop_id"] == codigo_desembarque
        ]["stop_sequence"].dropna().tolist()

        # Verificar se existe alguma combinação na ordem correta
        combinacoes = [
            (origem, destino)
            for origem in sequencias_embarque
            for destino in sequencias_desembarque
            if origem < destino
        ]

        if combinacoes:

            seq_embarque, seq_desembarque = min(
                combinacoes,
                key=lambda x: x[1] - x[0]
            )

            resultados.append({
                "Codigo_linha": linha["Codigo_linha"],
                "Letreiro": letreiro,
                "Sentido": linha["Sentido"],
                "Trip_ID": viagem["trip_id"],
                "Service_ID": viagem["service_id"],
                "Direction_ID": viagem["direction_id"],
                "Embarque": linha["Embarque"],
                "Desembarque": linha["Desembarque"],
                "Sequencia_embarque": int(seq_embarque),
                "Sequencia_desembarque": int(seq_desembarque)
            })

    return resultados

In [14]:
# Identificar serviços ativos na data atual
servicos_ativos = obter_servicos_ativos()

resultados_gtfs = []

# Validar cada linha geograficamente compatível
for _, linha in df_linhas_compativeis.iterrows():

    resultado = validar_trajeto_gtfs(
        linha,
        servicos_ativos
    )

    resultados_gtfs.extend(resultado)

# Organizar as viagens aprovadas
df_viagens_compativeis = pd.DataFrame(resultados_gtfs)

if not df_viagens_compativeis.empty:

    print("Viagens diretas encontradas:")
    print(len(df_viagens_compativeis))

    display(df_viagens_compativeis)

else:

    print("Nenhuma viagem direta programada foi encontrada.")

Viagens diretas encontradas:
8


,Codigo_linha,Letreiro,Sentido,Trip_ID,Service_ID,Direction_ID,Embarque,Desembarque,Sequencia_embarque,Sequencia_desembarque
0,33206,8622-10,2,8622-10-1,USD,1,ANGELICA C/B,PARADA PALMEIRAS C/B,7,13
1,34107,8615-10,2,8615-10-1,USD,1,ANGELICA C/B,PARADA PALMEIRAS C/B,11,17
2,34033,8400-10,2,8400-10-1,USD,1,ANGELICA C/B,PARADA PALMEIRAS C/B,7,13
3,35274,8000-1,2,8000-1-1,USD,1,ANGELICA C/B,PARADA PALMEIRAS C/B,6,12
4,33354,938C-10,2,938C-10-1,US_,1,ANGELICA C/B,PARADA PALMEIRAS C/B,6,12
5,33348,8549-10,2,8549-10-1,USD,1,ANGELICA C/B,PARADA PALMEIRAS C/B,7,13
6,33214,8686-10,2,8686-10-1,US_,1,ANGELICA C/B,PARADA PALMEIRAS C/B,6,12
7,33211,8677-10,2,8677-10-1,USD,1,ANGELICA C/B,PARADA PALMEIRAS C/B,6,12


In [15]:
# Relacionar viagens válidas com as distâncias geográficas

colunas_geograficas = [
    "Codigo_linha",
    "Codigo_embarque",
    "Codigo_desembarque",
    "Distancia_embarque",
    "Distancia_desembarque"
]

df_opcoes = df_viagens_compativeis.merge(
    df_linhas_compativeis[colunas_geograficas],
    on="Codigo_linha",
    how="inner"
)

# Calcular distância total de caminhada
df_opcoes["Distancia_caminhada"] = (
    df_opcoes["Distancia_embarque"] +
    df_opcoes["Distancia_desembarque"]
)

# Recuperar quantidade de veículos monitorados
monitoramento = df_candidatas[
    ["Codigo_linha", "Quantidade_veiculos"]
].rename(columns={
    "Quantidade_veiculos": "Veiculos_monitorados"
})

df_opcoes = df_opcoes.merge(
    monitoramento,
    on="Codigo_linha",
    how="left"
)

# Ordenar pelas prioridades de seleção
df_opcoes = df_opcoes.sort_values(
    ["Distancia_caminhada", "Veiculos_monitorados"],
    ascending=[True, False]
)

# Manter uma viagem representativa de cada linha
df_opcoes = df_opcoes.drop_duplicates(
    subset=["Codigo_linha"],
    keep="first"
)

print("OPÇÕES DE TRANSPORTE DISPONÍVEIS")

display(
    df_opcoes[
        [
            "Letreiro",
            "Sentido",
            "Embarque",
            "Desembarque",
            "Distancia_caminhada",
            "Veiculos_monitorados",
            "Trip_ID"
        ]
    ]
)

# Selecionar a primeira opção
if not df_opcoes.empty:

    linha_recomendada = df_opcoes.iloc[0].copy()

    print("\nLINHA RECOMENDADA:")
    print(linha_recomendada["Letreiro"])

else:

    linha_recomendada = None

    print("Nenhuma viagem compatível encontrada.")

OPÇÕES DE TRANSPORTE DISPONÍVEIS


,Letreiro,Sentido,Embarque,Desembarque,Distancia_caminhada,Veiculos_monitorados,Trip_ID
5,8549-10,2,ANGELICA C/B,PARADA PALMEIRAS C/B,50.42,5,8549-10-1
7,8677-10,2,ANGELICA C/B,PARADA PALMEIRAS C/B,50.42,5,8677-10-1
0,8622-10,2,ANGELICA C/B,PARADA PALMEIRAS C/B,50.42,4,8622-10-1
2,8400-10,2,ANGELICA C/B,PARADA PALMEIRAS C/B,50.42,3,8400-10-1
3,8000-1,2,ANGELICA C/B,PARADA PALMEIRAS C/B,50.42,3,8000-1-1
1,8615-10,2,ANGELICA C/B,PARADA PALMEIRAS C/B,50.42,2,8615-10-1
4,938C-10,2,ANGELICA C/B,PARADA PALMEIRAS C/B,50.42,2,938C-10-1
6,8686-10,2,ANGELICA C/B,PARADA PALMEIRAS C/B,50.42,2,8686-10-1



LINHA RECOMENDADA:
8549-10


In [16]:
def converter_horario(horario):
    horas, minutos = map(int, horario.split(":"))
    return horas * 60 + minutos

In [17]:
from IPython.display import display, Markdown


def consultar_previsao(codigo_linha, codigo_parada):

    resposta = sessao_api.get(
        f"{BASE_URL}/Previsao",
        params={
            "codigoLinha": int(codigo_linha),
            "codigoParada": int(codigo_parada)
        },
        timeout=30
    )

    # Renovar autenticação se necessário
    if resposta.status_code == 401:

        print("Renovando autenticação da SPTrans...")

        autenticar()

        resposta = sessao_api.get(
            f"{BASE_URL}/Previsao",
            params={
                "codigoLinha": int(codigo_linha),
                "codigoParada": int(codigo_parada)
            },
            timeout=30
        )

    resposta.raise_for_status()

    dados = resposta.json()
    registros = []

    if isinstance(dados, dict):

        parada = dados.get("p")

        if isinstance(parada, dict):

            for linha in parada.get("l") or []:

                if str(linha.get("cl")) != str(codigo_linha):
                    continue

                for veiculo in linha.get("vs") or []:

                    horario = veiculo.get("t")

                    if not horario:
                        continue

                    espera = None

                    if dados.get("hr"):

                        diferenca = (
                            converter_horario(horario)
                            - converter_horario(dados["hr"])
                        )

                        if diferenca < -720:
                            diferenca += 1440
                        elif diferenca > 720:
                            diferenca -= 1440

                        if diferenca >= 0:
                            espera = diferenca

                    registros.append({
                        "Prefixo": veiculo.get("p"),
                        "Previsao": horario,
                        "Espera_minutos": espera,
                        "Acessivel": veiculo.get("a"),
                        "Referencia": dados.get("hr")
                    })

    return pd.DataFrame(registros)


# ==========================================
# APRESENTAÇÃO DA RECOMENDAÇÃO
# ==========================================

if linha_recomendada is not None:

    linha = linha_recomendada

    distancia_embarque = float(linha["Distancia_embarque"])
    distancia_desembarque = float(linha["Distancia_desembarque"])

    # Apresentar informações do trajeto
    display(Markdown(f"""
# 🚌 Sua recomendação de viagem

**Linha recomendada: {linha['Letreiro']}**

Sentido de operação: {linha['Sentido']}

<|--|>

### 📍 Seu percurso

| Etapa | Localização | Distância |
|---|---|---:|
| Origem → Embarque | {linha['Embarque']} | {distancia_embarque:.1f} m |
| Desembarque → Destino | {linha['Desembarque']} | {distancia_desembarque:.1f} m |

**Distância total de caminhada: {distancia_embarque + distancia_desembarque:.1f} metros**

### ✅ Validação da viagem

- Viagem direta identificada.
- Sequência das paradas confirmada pelo GTFS.
- Viagem programada: `{linha['Trip_ID']}`.
""".replace("<|--|>", "---")))

    # Consultar previsão em tempo real
    try:

        df_previsao_final = consultar_previsao(
            linha["Codigo_linha"],
            linha["Codigo_embarque"]
        )

        if not df_previsao_final.empty:

            df_previsao_final = df_previsao_final.sort_values(
                "Espera_minutos",
                na_position="last"
            )

            proximo = df_previsao_final.iloc[0]

            acessibilidade = (
                "Sim" if proximo["Acessivel"] is True
                else "Não" if proximo["Acessivel"] is False
                else "Não informado"
            )

            espera = (
                f"{int(proximo['Espera_minutos'])} minutos"
                if pd.notna(proximo["Espera_minutos"])
                else "Não foi possível calcular"
            )

            display(Markdown(f"""
### ⏱ Próximo ônibus

| Informação | Resultado |
|---|---|
| Prefixo do veículo | **{proximo['Prefixo']}** |
| Previsão de chegada | **{proximo['Previsao']}** |
| Tempo estimado de espera | **{espera}** |
| Acessível para PcD | **{acessibilidade}** |

<small>Horário de referência da consulta: {proximo['Referencia']}</small>
"""))

            if len(df_previsao_final) > 1:

                print("\nOutros ônibus previstos:")

                display(
                    df_previsao_final.iloc[1:4][
                        [
                            "Prefixo",
                            "Previsao",
                            "Espera_minutos"
                        ]
                    ].rename(columns={
                        "Prefixo": "Veículo",
                        "Previsao": "Chegada prevista",
                        "Espera_minutos": "Espera (min)"
                    }).reset_index(drop=True)
                )

        else:

            display(Markdown("""
### ⏱ Previsão de chegada

A SPTrans não disponibilizou previsões de chegada para este momento.

**A recomendação de viagem continua válida**, pois a sequência programada foi confirmada pelo GTFS.
"""))

    except requests.exceptions.RequestException as erro:

        display(Markdown(f"""
### ⚠️ Previsão temporariamente indisponível

Não foi possível consultar os horários em tempo real.

Detalhe técnico: `{erro}`

A recomendação programada foi preservada.
"""))

else:

    display(Markdown("""
# Não encontramos uma viagem direta

Não foi possível identificar uma linha com todos os critérios necessários para o deslocamento informado.

Experimente outras coordenadas ou um raio de proximidade diferente.
"""))


# 🚌 Sua recomendação de viagem

**Linha recomendada: 8549-10**

Sentido de operação: 2

---

### 📍 Seu percurso

| Etapa | Localização | Distância |
|---|---|---:|
| Origem → Embarque | ANGELICA C/B | 20.9 m |
| Desembarque → Destino | PARADA PALMEIRAS C/B | 29.5 m |

**Distância total de caminhada: 50.4 metros**

### ✅ Validação da viagem

- Viagem direta identificada.
- Sequência das paradas confirmada pelo GTFS.
- Viagem programada: `8549-10-1`.



### ⏱ Previsão de chegada

A SPTrans não disponibilizou previsões de chegada para este momento.

**A recomendação de viagem continua válida**, pois a sequência programada foi confirmada pelo GTFS.
